## Data cleanning and preparation

In [470]:
import pandas as pd

companies = pd.read_csv("../data/raw/linkedin-job-postings/companies/companies.csv")
salaries = pd.read_csv("../data/raw/linkedin-job-postings/jobs/salaries.csv")
postings = pd.read_csv(
    "../data/processed/postings_clean.csv",
    engine="python",
    sep=",",
    quotechar='"',
    escapechar="\\",
    on_bad_lines="skip"   # o "warn" para ver cuántas saltas
)

## Handling missing values

Too many nulls in company_size in companies -> Missing values in `company_size` are replaced with an explicit "Unknown" category to preserve records while clearly indicating unavailable information.

In [471]:
# company_size: keep numeric, use NULL for missing
companies['company_size'] = (
    companies['company_size']
    .replace('Unknown', pd.NA)
    .astype('Int64')  # entero nullable
)

In [472]:
companies['company_size'].value_counts(dropna=False)

company_size
2       4956
1       4348
5       3918
3       3108
<NA>    2774
4       2333
7       1953
6       1083
Name: count, dtype: Int64

In [473]:
companies['company_size'].isna().sum()

np.int64(2774)

In [474]:
companies[companies['company_size'] == 'Unknown'].head(10)

,company_id,name,description,company_size,state,country,city,zip_code,address,url


In [475]:
companies.isna().mean().sort_values(ascending=False)

company_size    0.113349
description     0.012136
zip_code        0.001144
state           0.000899
address         0.000899
name            0.000041
city            0.000041
company_id      0.000000
country         0.000000
url             0.000000
dtype: float64

Too many nulls in min_salary, med_salary y max_salary in salaries -> Salary information is often partially reported across `min_salary`, `med_salary`, and `max_salary`. To maximize data usability without imputing values, a unified
`salary_value` field is created by prioritizing median salary when available, followed by minimum and maximum values.

In [476]:
salaries_clean = salaries.copy()

salaries_clean['salary_value'] = (
    salaries_clean['med_salary']
    .combine_first(salaries_clean['min_salary'])
    .combine_first(salaries_clean['max_salary'])
)

In [477]:
salaries_clean['salary_value'].isna().mean()

np.float64(0.0)

In [478]:
salaries_clean[
    ['min_salary', 'med_salary', 'max_salary', 'salary_value']
].head(10)

,min_salary,med_salary,max_salary,salary_value
0,NaN,20.00,NaN,20.00
1,23.0,NaN,25.0,23.00
2,100000.0,NaN,120000.0,100000.00
3,10000.0,NaN,200000.0,10000.00
4,33.0,NaN,35.0,33.00
5,NaN,48.43,NaN,48.43
6,50000.0,NaN,80000.0,50000.00
7,84000.0,NaN,101000.0,84000.00
8,50.0,NaN,55.0,50.00
9,72000.0,NaN,100000.0,72000.00


Normalize salary_value to YEARLY

In [479]:
PAY_PERIOD_TO_YEARLY = {
    'YEARLY': 1,
    'MONTHLY': 12,
    'WEEKLY': 52,
    'BIWEEKLY': 26,
    'HOURLY': 40 * 52
}

salaries_clean = salaries_clean.copy()

salaries_clean['salary_yearly'] = (
    salaries_clean['salary_value']
    * salaries_clean['pay_period'].map(PAY_PERIOD_TO_YEARLY)
)

In [480]:
salaries_clean[
    ['salary_value', 'pay_period', 'salary_yearly']
].head(30)

,salary_value,pay_period,salary_yearly
0,20.00,HOURLY,41600.0
1,23.00,HOURLY,47840.0
2,100000.00,YEARLY,100000.0
3,10000.00,YEARLY,10000.0
4,33.00,HOURLY,68640.0
5,48.43,HOURLY,100734.4
6,50000.00,YEARLY,50000.0
7,84000.00,YEARLY,84000.0
8,50.00,HOURLY,104000.0
9,72000.00,YEARLY,72000.0


In [481]:
salaries_clean.isna().mean().sort_values(ascending=False)

med_salary           0.83234
max_salary           0.16766
min_salary           0.16766
salary_id            0.00000
job_id               0.00000
pay_period           0.00000
currency             0.00000
compensation_type    0.00000
salary_value         0.00000
salary_yearly        0.00000
dtype: float64

Too many nulls in closed_time in postings -> The `closed_time` field contains many missing values. This is expected behavior, as the field represents an event (job closing) that may not have occurred. Missing values are therefore interpreted as postings that are not explicitly marked as closed. An additional boolean field `is_closed` is created to explicitly capture the closure status of job postings.

In [482]:
postings_clean = postings.copy()

postings_clean['is_closed'] = postings_clean['closed_time'].notna()

In [483]:
postings_clean['is_closed'].value_counts()

is_closed
False    122820
True       1076
Name: count, dtype: int64

In [484]:
postings.isna().mean().sort_values(ascending=False)

closed_time                   0.991315
skills_desc                   0.980298
med_salary                    0.949280
remote_allowed                0.876929
applies                       0.811761
min_salary                    0.759516
max_salary                    0.759500
currency                      0.708861
normalized_salary             0.708861
compensation_type             0.708861
pay_period                    0.708804
posting_domain                0.323029
application_url               0.296361
formatted_experience_level    0.237804
fips                          0.221710
zip_code                      0.168900
company_id                    0.014318
company_name                  0.014181
views                         0.014044
sponsored                     0.000452
expiry                        0.000444
listed_time                   0.000444
original_listed_time          0.000436
formatted_work_type           0.000428
job_posting_url               0.000428
application_type         

## Fixing invalid placeholders, Column selection and standardization, Fields kept as-is and Final cleaned datasets

Registers detected with `country`, `state`, `zipcode`  field containing `"0"`, which is not valid null in companies -> Some records contained the value "0" in geographic fields (`country`, `state`, and `zip_code`), which does not represent valid information. These placeholders were replaced with proper missing values to ensure consistent data quality.

In [485]:
companies[['country', 'state', 'zip_code']].value_counts().head()

country  state       zip_code
0        0           0           715
US       0           0           597
         California  0           140
         Texas       0           110
         CA          0           109
Name: count, dtype: int64

In [486]:
companies_clean = companies.copy()

cols_with_invalid_zero = ['country', 'state', 'zip_code']

companies_clean[cols_with_invalid_zero] = (
    companies_clean[cols_with_invalid_zero]
    .replace('0', pd.NA)
)

In [487]:
companies_clean[['country', 'state', 'zip_code']].isna().mean()

country     0.029706
state       0.089772
zip_code    0.126180
dtype: float64

Registers detected with `address` field containing `"."` and `"-"`, which is not valid nul in companies -> The `address` field contained invalid placeholder values such as "." and "-", which do not represent real addresses. These values were replaced with proper missing values to ensure consistent handling of missing data.

In [488]:
companies_clean['address'] = companies_clean['address'].replace(
    {'.': pd.NA, '-': pd.NA}
)


In [489]:
companies_clean['address'].isna().mean()

np.float64(0.0013892861520859723)

In [490]:
companies_clean['address'].value_counts().head()

address
0                     3972
New York                 6
433 W Van Buren St       6
Downtown                 6
175 Greenwich St         5
Name: count, dtype: int64

In [491]:
(companies_clean['address'].isin(['.', '-'])).sum()

np.int64(0)

Two columns describing job type are available: `work_type` and `formatted_work_type`. Both convey similar information; however, `formatted_work_type` provides a cleaner and more standardized representation. Therefore, it is used as the primary field for analysis, while `work_type` is excluded from downstream usage. The `work_type` column was removed from the cleaned dataset, as a standardized version (`formatted_work_type`) is used for analysis.

In [492]:
postings_clean = postings.copy()

postings_clean = postings_clean.drop(columns=['work_type'], errors='ignore')

In [493]:
# ------------------------------
# Fix invalid company_id (FK-safe)
# ------------------------------

# Normalizar tipos (vienen como float)
companies_clean["company_id"] = (
    pd.to_numeric(companies_clean["company_id"], errors="coerce")
    .astype("Int64")
)

postings_clean["company_id"] = (
    pd.to_numeric(postings_clean["company_id"], errors="coerce")
    .astype("Int64")
)

# job_id es obligatorio para la tabla (PK). Si no existe, se descarta la fila.
postings_for_db["job_id"] = pd.to_numeric(postings_for_db["job_id"], errors="coerce")
before = len(postings_for_db)
postings_for_db = postings_for_db.dropna(subset=["job_id"]).copy()
postings_for_db["job_id"] = postings_for_db["job_id"].astype("int64")
print("Dropped rows without job_id:", before - len(postings_for_db))

valid_company_ids = set(companies_clean["company_id"].dropna())

# Si company_id no existe en companies -> NULL
postings_clean.loc[
    ~postings_clean["company_id"].isin(valid_company_ids),
    "company_id"
] = pd.NA


Dropped rows without job_id: 56


In [494]:
postings_clean.columns

Index(['job_id', 'company_name', 'title', 'description', 'max_salary',
       'pay_period', 'location', 'company_id', 'views', 'med_salary',
       'min_salary', 'formatted_work_type', 'applies', 'original_listed_time',
       'remote_allowed', 'job_posting_url', 'application_url',
       'application_type', 'expiry', 'closed_time',
       'formatted_experience_level', 'skills_desc', 'listed_time',
       'posting_domain', 'sponsored', 'currency', 'compensation_type',
       'normalized_salary', 'zip_code', 'fips'],
      dtype='object')

In [495]:
postings_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123896 entries, 0 to 123895
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123896 non-null  object 
 1   company_name                122139 non-null  object 
 2   title                       123857 non-null  object 
 3   description                 123845 non-null  object 
 4   max_salary                  29797 non-null   object 
 5   pay_period                  36078 non-null   object 
 6   location                    123846 non-null  object 
 7   company_id                  122122 non-null  Int64  
 8   views                       122156 non-null  object 
 9   med_salary                  6284 non-null    object 
 10  min_salary                  29795 non-null   object 
 11  formatted_work_type         123843 non-null  object 
 12  applies                     23322 non-null   object 
 13  original_liste

## Save cleaned datasets
All cleaned datasets were exported to the `processed/` directory and will be used
as the input for subsequent analysis phases.

In [496]:
# ------------------------------
# Prepare postings for database (schema-aligned)
# ------------------------------

cols_job_postings = [
    "job_id",
    "company_id",
    "company_name",
    "title",
    "description",
    "location",
    "zip_code",
    "fips",
    "formatted_work_type",
    "formatted_experience_level",
    "remote_allowed",
    "job_posting_url",
    "application_url",
    "application_type",
    "posting_domain",
    "views",
    "applies",
    "original_listed_time",
    "listed_time",
    "expiry",
    "closed_time",
    "sponsored",
    "skills_desc"
]

postings_for_db = postings_clean[
    [c for c in cols_job_postings if c in postings_clean.columns]
].copy()


In [497]:
from pathlib import Path

PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

companies_clean.to_csv(PROCESSED_PATH / "companies_clean.csv", index=False)
postings_clean.to_csv(PROCESSED_PATH / "postings_clean.csv", index=False)
salaries_clean.to_csv(PROCESSED_PATH / "salaries_clean.csv", index=False)

import csv

# 1) Limpieza defensiva de texto (evita caracteres de control raros)
text_cols = ["company_name","title","description","location","job_posting_url",
             "application_url","application_type","posting_domain","skills_desc"]

for c in text_cols:
    if c in postings_for_db.columns:
        postings_for_db[c] = (
            postings_for_db[c]
            .astype("string")
            # elimina caracteres de control excepto \n y \t (opcional)
            .str.replace(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", " ", regex=True)
        )

# 2) Fuerza numéricos/booleanos para que el CSV salga consistente
num_cols = ["job_id","company_id","views","applies","original_listed_time","listed_time","expiry","closed_time"]
for c in num_cols:
    if c in postings_for_db.columns:
        postings_for_db[c] = pd.to_numeric(postings_for_db[c], errors="coerce")


if "remote_allowed" in postings_for_db.columns:
    ra = postings_for_db["remote_allowed"]

    # Pásalo a string, limpia espacios, y normaliza
    ra = ra.astype("string").str.strip().str.lower()

    # Mapeo amplio de valores posibles
    mapping = {
        "true": True, "t": True, "1": True, "yes": True, "y": True,
        "false": False, "f": False, "0": False, "no": False, "n": False,
        "": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA
    }

    postings_for_db["remote_allowed"] = ra.map(mapping).astype("boolean")


# 3) Export CSV a prueba de bombas
postings_for_db.to_csv(
    PROCESSED_PATH / "postings_for_db.csv",
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    escapechar="\\",
    lineterminator="\n"
)
